In [1]:
from langchain_openai import ChatOpenAI

In [2]:
from langchain.agents import create_agent

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

In [4]:
from langchain.tools import tool

In [5]:
from typing import List, Union, Optional, Dict, Literal

In [6]:
import tiktoken

In [7]:
import os
from dotenv import load_dotenv

In [8]:
load_dotenv()

True

In [9]:
from pathlib import Path

In [10]:
llm = ChatOpenAI(
    base_url= "http://localhost:5000/gateway/mlflow/v1",#os.getenv("BASE_URL"),
    api_key="my-api-key",
    model="my-endpoint",#os.getenv("LLM"),
)

In [11]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calculator(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

In [12]:
tools = [calculator]
tool_map = {tool.name: tool for tool in tools}

In [13]:
import json

In [75]:
class InternalTools:
    @classmethod
    def tools(cls):
        @tool("collapsed_tool_result", description="Fetch old collapsed tool result using tool call id.")
        def collapsed_tool_result(tool_call_id: str) -> str:
            # Ensure path is a Path object and join it with the filename
            file_path = Thread.get_tool_result_path() / tool_call_id
            
            try:
                # Direct, clean reading using pathlib
                return file_path.read_text(encoding="utf-8")
            except Exception as e:
                return str(e)

        return [collapsed_tool_result]


In [76]:
class Agent:
    def __init__(self, model: ChatOpenAI, tools=List):
        self.model = model
        self.tools = tools
        if self.tools:
            self.tools.extend(InternalTools.tools())
            self.model = self.model.bind_tools(self.tools)
        self.tool_map = {tool.name: tool for tool in self.tools}
            
    def invoke(self, thread: Thread, self_append: bool = True):
        if thread.tail is not None:
            thread = thread.tail
        thread.agent = self
        while True:
            response = self.model.invoke(thread.messages)
            
            if not self_append:
                thread.agent = None
                return response
                
            thread.append(response)
            if response.tool_calls:
                for tool in response.tool_calls:
                    args=tool["args"]
                    call_id = tool["id"]
                    name = tool["name"]
                    result = self.tool_map[name].invoke(args)
                    thread.append(ToolMessage(name=name, content=result, tool_call_id=call_id))
            else:
                thread.agent = None
                return response

    def __ror__(self, thread: Thread):
        return self.invoke(thread)

In [77]:
class ThreadHideRule:
    def __init__(
        self,
        name: str,
        message: str,
    ):
        self.name = name
        self.message = message

In [96]:
class AutoToolHideRule:
    def __init__(
        self,
        token_limit: int,
        per_tool_token_limit: Optional[int] = None
    ):
        self.token_limit = token_limit
        self.per_tool_token_limit = per_tool_token_limit

In [97]:
class Thread:
    def __init__(
        self, 
        messages: List[Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]] = None, 
        system_prompt: Union[str, SystemMessage] = None,
        compression_prompt: str = None,
        token_limit: int = None,
        tool_hide_rules: List[Union[ThreadHideRule, AutoToolHideRule]] = None   
    ):
        self.messages = []
        self.system_prompt = system_prompt
        self.compression_prompt = compression_prompt
        self.token_limit = token_limit
        self.agent = None
        self.encoder = tiktoken.encoding_for_model("gpt-4o-mini")
        self.root = None
        self.parent = None
        self.child = None
        self.tail = None
        self.tool_hide_rules = tool_hide_rules
        self.path = Path.cwd() / "tool_results"
        self.path.mkdir(parents=True, exist_ok=True)

        
        if messages is not None:
            index = self._find_system_message(messages)
            if index == -1 or index == 0:
                self.messages = messages
            else:
                raise Valueerror(
                    f"system message not at the starting, it was found at {index} index"
                )
                    
        if self.system_prompt is not None:
            index = self._find_system_message(self.messages)
            if isinstance(self.system_prompt, str):
                self.system_prompt = SystemMessage(self.system_prompt)
            if index == 0:
                if len(self.messages) == 0:
                    self.append(self.system_prompt)
                else:
                    self[0] = self.system_prompt
            elif index == -1:
                self.messages = [self.system_prompt] + self.messages
            else:
                pass

    @classmethod
    def get_tool_result_path(cls):
        path = Path.cwd() / "tool_results"
        path.mkdir(parents=True, exist_ok=True)
        return path

    def _find_system_message(self, messages: List[Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]]):
        for i in range(len(messages)):
            if isinstance(messages[i], SystemMessage):
                return i
        return -1

    def count_token(self):
        if self.tail is not None:
            content = [m.content for m in self.tail]
        else:
            content = [m.content for m in self]
        merged = "\n".join(content)
        return len(self.encoder.encode(merged))

    def calculate_tokens(self, content):
        return len(self.encoder.encode(content))
        
        
    def append(self, message: Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]):
        if self.root is not None:
            tool_hide_rules = self.root.tool_hide_rules
        else:
            tool_hide_rules = self.tool_hide_rules

        thread_hide_rules = [rule for rule in tool_hide_rules if isinstance(rule, ThreadHideRule)]
        
        auto_tool_hide_rules = None
        for rule in tool_hide_rules:
            if isinstance(rule, AutoToolHideRule):
                auto_tool_hide_rules = rule
                break
        
            
        if isinstance(message, ToolMessage) and tool_hide_rules is not None:
            name = message.name
            match = False
            tool_hide_rule = None
            for rule in thread_hide_rules:
                if rule.name == name:
                    match = True
                    tool_hide_rule = rule
                    break
            if match:
                for m in reversed(self):
                    if isinstance(m, ToolMessage) and m.name == name:
                        self.save_tool_result(m)
                        m.content = tool_hide_rule.message + f"\n tool call id {m.tool_call_id}. use the ID to retrive this tool result using collapsed_tool_result"
                        break
                        
        if auto_tool_hide_rules is not None:
            total_tokens = self.count_token()
            if total_tokens >= auto_tool_hide_rules.token_limit:
                if auto_tool_hide_rules.per_tool_token_limit is not None:
                    per_tool_token_limit = auto_tool_hide_rules.per_tool_token_limit
                    for m in reversed(self):
                        if isinstance(m, ToolMessage) and self.calculate_tokens(m.content) > per_tool_token_limit:
                            self.save_tool_result(m)
                            m.content = f"This tool call result has been collapsed due to token size constraints.\n tool call id {m.tool_call_id}. use the ID to retrive this tool result using collapsed_tool_result"
                else:
                    per_tool_token_limit = 8_000
                    for m in reversed(self):
                        if isinstance(m, ToolMessage) and self.calculate_tokens(m.content) > per_tool_token_limit:
                            self.save_tool_result(m)
                            m.content = f"This tool call result has been collapsed due to token size constraints.\n tool call id {m.tool_call_id}. use the ID to retrive this tool result using collapsed_tool_result"

            
        

                        
        token_usuage = self.count_token()
        if self.token_limit is not None and token_usuage > self.token_limit and self.agent is not None and self.compression_prompt is not None:
            self.messages.append(HumanMessage(self.compression_prompt))
            compression_report = self.agent.invoke(self, self_append=False).content
            self.messages.pop()
            
            new_thread = self.copy()
            self.child = new_thread
            new_thread.parent = self
            new_thread.root = self.root if self.root is not None else self
            self.root.tail = new_thread

            new_thread.messages = []
            if isinstance(self.root[0], SystemMessage):
                new_thread.append(self.root[0])
            new_thread.append(HumanMessage(compression_report)) 
        else:
            self.messages.append(message)

    def save_tool_result(self, message: ToolMessage) -> bool:
        try:
            with open(self.path / str(message.tool_call_id), "w", encoding="utf-8") as f:
                f.write(message.content)
            return True
        except:
            return False

    def count(self):
        counts = {
            "depth":0,
            "system":0, 
            "human":0,
            "ai":0,
            "tool":0
        }
        thread = self
        depth = 0
        if isinstance(thread[0], SystemMessage):
            counts["system"]=1
        while True:
            for m in thread:
                if isinstance(m, AIMessage):
                    counts["ai"]+=1
                elif isinstance(m, HumanMessage):
                    counts["human"]+=1
                elif isinstance(m, ToolMessage):
                    counts["tool"]+=1
                else:
                    pass
            if thread.child is None:
                break
            else:
                thread = thread.child
                depth += 1
        counts["depth"] = depth
        return counts

    def __ror__(self, other: Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]):
        if self.tail is not None:
            self.tail.append(other)
        else:
            self.append(other)

    # def __add__(self, other: Thread):
    #     new_thread = Thread()
    #     new_thread.messages = self.messages + other.messages
    #     return new_thread

    def __str__(self):
        counts = self.count()
        return json.dumps(counts)

    def __repr__(self):
        counts = self.count()
        return json.dumps(counts)

    def __iter__(self):
        if self.tail is not None:
            for msg in self.tail.messages:
                yield msg
        else:
            for msg in self.messages:
                yield msg
                
    def __getitem__(self, index):
        if self.tail is not None:
            return self.tail.messages[index]
        else:
            return self.messages[index]

    def __len__(self):
        if self.tail is not None:
            return len(self.tail.messages)
        else:
            return len(self.messages)

    def __setitem__(self, index, value):
        if self.tail is not None:
            self.tail.messages[index] = value
        else:
            self.messages[index] = value

    def __copy__(self):
        new_instance = Thread()
        return new_instance
        

In [98]:
myagent = Agent(
    model=llm,
    tools=tools
)

In [80]:
thread = Thread(
    tool_hide_rules=[
        ThreadHideRule(
            name = "calculator",
            message = "Previous calculation result has been collapsed."
        ),
         AutoToolHideRule(
             token_limit = 20_000
         )
    ]
)

In [81]:
thread.append(SystemMessage("You are a Mathematical Expression solver"))

In [90]:
HumanMessage("what is the answer of the first query when using collapsed_tool_result") | thread

In [86]:
HumanMessage("calculate this expression : 414%4 using the calculator tool only") | thread

In [94]:
thread.messages

[SystemMessage(content='You are a Mathematical Expression solver', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='calculate this expression : 414+4 using the calculator tool only', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 524, 'total_tokens': 587, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Deepseek-vapt', 'system_fingerprint': None, 'id': 'chatcmpl-a2276f4a0ea3b89a', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0566b-05fc-7820-9603-4c77db9164b2-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '414+4'}, 'id': 'chatcmpl-tool-b71a64f7f7ba9c86', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 524, 'output_tokens': 63, 'total_tokens': 587, 'input_token_details': {}, 'output_token_details': {}}),
 ToolMes

In [92]:
response = thread | myagent

In [93]:
response

AIMessage(content='The answer of the first query (**414 + 4**) when retrieved using `collapsed_tool_result` is **418**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 827, 'total_tokens': 888, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Deepseek-vapt', 'system_fingerprint': None, 'id': 'chatcmpl-a1ddfe4c9b606fef', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0566b-daa7-7602-a3fe-466c3cf1be2c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 827, 'output_tokens': 61, 'total_tokens': 888, 'input_token_details': {}, 'output_token_details': {}})

In [65]:
response.content

'The result of **146 % 4** is **2**.'

In [41]:
print(thread)

{"depth": 0, "system": 1, "human": 3, "ai": 5, "tool": 2}


In [48]:
thread.messages

[SystemMessage(content='You are a Mathematical Expression solver', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the answer of the first query when using calculator tool', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I don\'t see any previous calculator tool calls in this conversation. This appears to be the start of our interaction, so there is no "first query" or prior calculator usage to reference.\n\nIf you\'d like me to use the calculator tool to solve a mathematical expression, please provide the expression you\'d like me to evaluate!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 198, 'prompt_tokens': 352, 'total_tokens': 550, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Deepseek-vapt', 'system_fingerprint': None, 'id': 'chatcmpl-bbc2078ee0b8472e', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a05662-ac7e-7

In [39]:
for m in response:
    if isinstance(m, SystemMessage):
        print("system message:\n")
    elif isinstance(m, HumanMessage):
        print("human message:\n")
    elif isinstance(m, AIMessage):
        print("ai message:\n")
    elif isinstance(m, ToolMessage):
        print("tool message:\n")
    else:
        print("other type:\n")
    print(m.model_dump_json())
    print("\n\n")

system message:

{"content":"You are a Mathematical Expression solver","additional_kwargs":{},"response_metadata":{},"type":"system","name":null,"id":null}



human message:

{"content":"Solve this particular expression : 3*4+77 use calculator tool","additional_kwargs":{},"response_metadata":{},"type":"human","name":null,"id":null}



ai message:

{"content":"","additional_kwargs":{"refusal":null},"response_metadata":{"token_usage":{"completion_tokens":70,"prompt_tokens":297,"total_tokens":367,"completion_tokens_details":null,"prompt_tokens_details":null},"model_provider":"openai","model_name":"Deepseek-vapt","system_fingerprint":"vllm-0.25.0-tp2-ep-d4f8ac0c","id":"chatcmpl-bc18df2e93c1cfe1","finish_reason":"tool_calls","logprobs":null},"type":"ai","name":null,"id":"lc_run--01a03256-a5d0-7433-8fa3-9299a904fa75-0","tool_calls":[{"name":"calculator","args":{"expression":"3*4+77"},"id":"chatcmpl-tool-8e2f16389057886f","type":"tool_call"}],"invalid_tool_calls":[],"usage_metadata":{"input_t

In [19]:
s = SystemMessage("hii")

In [20]:
s.model_dump_json()

'{"content":"hii","additional_kwargs":{},"response_metadata":{},"type":"system","name":null,"id":null}'

In [45]:
response["messages"][-1].content

'The result of the expression \\(3 \\times 4 + 77\\) is **89**.'

In [47]:
response["messages"].append(HumanMessage("Now also calculate this one 456%100"))

In [54]:
nr = agent.invoke(nr)